spark session.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    array_contains, col, coalesce, concat_ws, expr, length, lit,
)

# Same parquet-committer override as notebook 02 — cluster default points at an EMR
# class whose JAR isn't on the classpath.
spark = SparkSession.builder \
    .appName('FB_API_topics') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

Master: yarn
Spark version: 3.5.0


26/05/17 04:37:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


paths.

In [2]:
V2_PATH = '/user/s3348393/main/preprocessing/v2/parquet'

load v2 and filter to the residual corpus. filter out the duplicates and only keep match_type is null, and in english. Ignore some commercial entities found while working through the problem up front. 

the body text etc are arrays - pull out the first non-null value from each array (probably the first, but just being careful)

some of the bodies are empty - add in the title and description to try and up the number of rows we can use

In [19]:
df = spark.read.parquet(V2_PATH)
print('All v2 rows:    ', df.count())


def first_non_empty(col_name):
    """First non-null, non-empty element of an array column. Returns null if none."""
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")


#rejects
COMMERCIAL_BYLINES = {
    'Access',                            # Indigenous Employment Australia — job listings
    'Streamotion Pty Ltd',               # Kayo / Binge sports + entertainment streaming
    'SBS Australia',                     # SBS On Demand streaming promotions
    'SBS Arabic24',                      # SBS language-stream marketing
    'SBS Mandarin中文普通话',            # SBS language-stream marketing
    'The Squiz',                         # paid news newsletter
    'Hair Cooki',                           #hair care ad
    "Shell"
}


corpus = df.filter(
        (col('ad_seq_no') == 1) &
        col('match_type').isNull() &
        # most of the language values are null - keep nulls and english
        (col('languages').isNull() | array_contains('languages', 'en')) &
        ~col('bylines').isin(list(COMMERCIAL_BYLINES))
    ) \
    .withColumn('body_text',  first_non_empty('creative_bodies')) \
    .withColumn('desc_text',  first_non_empty('creative_link_descs')) \
    .withColumn('title_text', first_non_empty('creative_link_titles')) \
    .withColumn('body',
        concat_ws(' ',
            coalesce(col('body_text'),  lit('')),
            coalesce(col('desc_text'),  lit('')),
            coalesce(col('title_text'), lit('')),
        )
    ) \
    .filter(length(col('body')) > 0) \
    .drop('body_text', 'desc_text', 'title_text')

print('Residual corpus:', corpus.count())
corpus.select('page_name', 'bylines', 'body').show(3, truncate=80)

All v2 rows:     5796491


Residual corpus: 94358
+----------------------------------+-----------------------------+--------------------------------------------------------------------------------+
|                         page_name|                      bylines|                                                                            body|
+----------------------------------+-----------------------------+--------------------------------------------------------------------------------+
|                    Thrive by Five|               Thrive By Five|Our early learning and childcare centres are under stress, and early childhoo...|
|Time for Change - Change Aged Care|         United Workers Union|Important Information for Aged Care workers. \n\nYour union has compiled a li...|
|     Australian Ethical Investment|Australian Ethical Investment|The sooner you switch to animal-friendly super, the sooner factory farming an...|
+----------------------------------+-----------------------------+-----------------------

add to the stop words some generic keywords that kept popping up. 

In [13]:
from pyspark.ml.feature import StopWordsRemover

stop_words = StopWordsRemover.loadDefaultStopWords('english') + [
    # URL / web junk that survives tokenisation
    'https', 'http', 'www', 'com', 'org', 'au', 'co', 'html',
    # contraction fragments surviving minTokenLength=2
    're', 've', 'll',
    # generic fillers (high frequency, low topic-discrimination value)
    'help', 'time', 'like', 'need', 'make', 'take', 'people',
    'year', 'years', 'today', 'also', 'will', 'can', 'get',
    'see', 'know', 'one', 'two', 'new', 'now', 'us',
    # generic CTA (kept short — don't strip 'petition', 'donate', 'sign', etc.
    # since those carry topic signal)
    'click', 'learn',
    #others
    'australia', 'australian', "2022"
]

print('Stop-words list size:', len(stop_words))

Stop-words list size: 218


preprog pipeline. 

regex tokenizer, 
stop word remover
count vectoriser. 

re-run the pipeline a few times to tune the count vectoriser.

In [14]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, CountVectorizer

tokenizer = RegexTokenizer(
    inputCol='body', outputCol='raw_tokens',
    pattern=r'\W+', toLowercase=True, minTokenLength=2,
)

remover = StopWordsRemover(
    inputCol='raw_tokens', outputCol='tokens',
    stopWords=stop_words,
)

vectorizer = CountVectorizer(
    inputCol='tokens', outputCol='features',
    vocabSize=5000, #was 10000
    minDF=100,    # was 50
    maxDF=0.3,    # seems ok?
)

prep_pipeline = Pipeline(stages=[tokenizer, remover, vectorizer])

i did a bunch of runs, and apparently cache should help.

In [15]:
prep_model  = prep_pipeline.fit(corpus)
features_df = prep_model.transform(corpus).cache()

vocab = prep_model.stages[-1].vocabulary
print('Vocabulary size:', len(vocab))
print('\nTop 30 vocabulary terms (most frequent first):')
for i in range(0, 30, 3):
    print(f"{vocab[i]:<18}{vocab[i+1]:<18}{vocab[i+2]}")

Vocabulary size: 4251

Top 30 vocabulary terms (most frequent first):
sign              climate           government
support           community         petition
change            vote              world
protect           women             future
action            local             join
free              election          stop
woodside          share             council
children          life              donate
energy            every             labor
gas               tell              health


test some different group sizes on 10% data sample.

In [17]:
from pyspark.ml.clustering import LDA

#10%
sample_df = features_df.sample(0.1, seed=42).cache()

# Lookup from CountVectorizer integer term indices back to readable words.
vocab = prep_model.stages[-1].vocabulary

# Fit LDA at each k. seed=42 fixed so k=5 vs k=10 etc. are comparable.
for k in [5, 10, 15, 20, 25, 30]:
    print(k)
    lda = LDA(featuresCol='features', k=k, maxIter=20, seed=42)
    model = lda.fit(sample_df)
    topics = model.describeTopics(maxTermsPerTopic=10).collect()
    for row in topics:
        words = ' '.join(vocab[i] for i in row.termIndices)
        print(f'  Topic {row.topic:>2}: {words}')

sample_df.unpersist()

5


26/05/17 04:46:51 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: children women support energy day violence many protect government home
  Topic  1: sign petition government woodside support demand tell enough share donate
  Topic  2: vote early school learning every election solar agl childcare education
  Topic  3: climate community government local sign council action support free change
  Topic  4: climate protect sign woodside change join gas future world save
10


26/05/17 04:47:00 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: violence domestic support stop children safe child abuse emergency women
  Topic  1: sign woodside enough petition tell share demand stop send gas
  Topic  2: early learning vote childcare families voting affordable every quiz many
  Topic  3: election local vote council abc community party independent liberal parliament
  Topic  4: climate protect gas donate woodside super stop project future species
  Topic  5: climate wrc industry forestry 50 forest sydney mayoral communities festival
  Topic  6: sign plastic use free single life share plastics support takes
  Topic  7: government support labor every families donate school morrison provide day
  Topic  8: climate community government energy action change support council health local
  Topic  9: women sign children oceans world petition protect ocean global rights
15


...

26/05/17 04:47:30 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: child sex name add sexual abuse trade pledge rescue deforestation
  Topic  1: kimberley richard river martuwarra issue oz aged care media test
  Topic  2: early learning childcare quiz families many affordable fair every petition
  Topic  3: production guy costs transport liberal wildlife demand ballarat heard glider
  Topic  4: super switch electricity educators future join animal welfare animals read
  Topic  5: wrc mayoral 50 1977 communities hood candidate forensic salary back
  Topic  6: plastic sign use single plastics share free big ban businesses
  Topic  7: laws quoll spotted quolls numbers tail living environment strong nature
  Topic  8: community school local state day council working government road team
  Topic  9: every de queensland minister auspol child environment la plibersek music
  Topic 10: film watch good causes feel movies tv care find queensland
  Topic 11: woodside gas sign project enough stop women tell whales fossil
  Topic 12: genus sustainabili

26/05/17 04:47:42 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: child sex name add sexual trade abuse pledge deforestation rescue
  Topic  1: kimberley river martuwarra richard issue media aged weekly care arab
  Topic  2: early learning childcare quiz families affordable many fair minute support
  Topic  3: production plastic ballarat double across 30th guy glider june cut
  Topic  4: super educators switch animal join welfare childhood animals ethical read
  Topic  5: wrc mayoral communities 50 1977 hood forensic candidate back salary
  Topic  6: sign plastic use single share support ukraine plastics conflict hunger
  Topic  7: laws quoll spotted quolls numbers tail environment strong nature researchers
  Topic  8: community school day state event working housing young local road
  Topic  9: every de auspol queensland child plibersek la music votethemout education
  Topic 10: wildlife mp nature protect religious queensland senate liberal best faith
  Topic 11: woodside sign enough women tell gas stop share message send
  Topic 12: gen

DataFrame[id: string, page_id: string, page_name: string, snapshot_date: date, ad_creation_date: date, ad_delivery_start_date: date, ad_delivery_stop_date: date, creative_bodies: array<string>, creative_link_captions: array<string>, creative_link_descs: array<string>, creative_link_titles: array<string>, spend_lower_bound: bigint, spend_upper_bound: bigint, spend_mid: double, impressions_lower_bound: bigint, impressions_upper_bound: bigint, impressions_mid: double, audience_size_lower_bound: bigint, audience_size_upper_bound: bigint, audience_size_mid: double, currency: string, languages: array<string>, publisher_platforms: array<string>, demographic_distribution: array<struct<age:string,gender:string,percentage:string>>, delivery_by_region: array<struct<percentage:string,region:string>>, ad_snapshot_url: string, bylines: string, ad_seq_no: int, match_type: string, political_party: string, body: string, raw_tokens: array<string>, tokens: array<string>, features: vector]


below 20 blurs, above 20 some of the bags dont make sense. at 20 we can clearly see some topic clusters which dont look political, and can remove them.

In [ ]:
from pyspark.ml.clustering import LDA
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

K = 20

# Fit LDA at k=K on the full cached features. seed=42 for reproducibility.
lda = LDA(featuresCol='features', k=K, maxIter=20, seed=42)
lda_model = lda.fit(features_df)

# Transform: attach topicDistribution and dominant topic_id to every ad.
classified = lda_model.transform(features_df) \
    .withColumn('topic_array', vector_to_array('topicDistribution')) \
    .withColumn('topic_id', expr('array_position(topic_array, array_max(topic_array)) - 1'))

# Vocabulary for term-index -> word translation.
vocab = prep_model.stages[-1].vocabulary

# Top 15 terms per topic from describeTopics.
topics_rows = lda_model.describeTopics(maxTermsPerTopic=15).collect()
topic_terms = {row.topic: [vocab[i] for i in row.termIndices] for row in topics_rows}

# Top 5 bylines per topic — Spark window function over (topic_id, bylines, count).
w = Window.partitionBy('topic_id').orderBy(desc('count'))
top_bylines_rows = classified.filter(col('bylines').isNotNull()) \
    .groupBy('topic_id', 'bylines').count() \
    .withColumn('rank', row_number().over(w)) \
    .filter(col('rank') <= 5) \
    .orderBy('topic_id', 'rank') \
    .collect()

top_bylines = {}
for row in top_bylines_rows:
    top_bylines.setdefault(row.topic_id, []).append(row.bylines)

# Print to console — easy to scan while you draft labels.
print(f'k = {K}\n')
for tid in range(K):
    words = ' '.join(topic_terms.get(tid, []))
    bls   = ' | '.join(top_bylines.get(tid, []))
    print(f'Topic {tid:>2}:')
    print(f'  Terms:   {words}')
    print(f'  Bylines: {bls}')
    print()